# UMAP Diagnostic — 6 drugs (site / species / resistance)

In [ ]:
# ── CONFIG ──
ALL_DRUGS = [
    "Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic_acid",
    "Piperacillin-Tazobactam", "Ceftriaxone", "Ceftazidime"
]
SITE_ORDER = ["A", "B", "C", "D"]
MASK_TOP_K = 500

# UMAP parameters
UMAP_N_NEIGHBORS = 20
UMAP_MIN_DIST = 0.25
UMAP_N_COMPONENTS = 2
UMAP_RANDOM_STATE = 42

# Subsampling (stratified by site) to keep UMAP tractable + readable
GLOBAL_SUBSAMPLE = 20000
PER_DRUG_SUBSAMPLE = 8000

RUN_NAME = "UMAP-Diagnostic"


In [ ]:
!pip install umap-learn --quiet


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import warnings, json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import umap
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_context("notebook")
print("Imports ready. umap-learn", umap.__version__)


In [ ]:
# ── Paths ──
if IN_COLAB:
    BASE = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    BASE = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")

ANALYSIS_DIR = BASE / "Processing/Analysis"
SHARED_MASKS_DIR = ANALYSIS_DIR / "08-Federated-mlp-lr-rf" / "Species-Masking" / "shared_masks"
OUT_DIR = ANALYSIS_DIR / "08-Federated-mlp-lr-rf" / "Species-Masking" / RUN_NAME / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Masks dir: {SHARED_MASKS_DIR}")
print(f"Output dir: {OUT_DIR}")


In [ ]:
# ── Load data (6 drugs x 4 sites) ──
def load_drug(drug):
    codes, Xs, ys, sps, sites = [], [], [], [], []
    for si, site in enumerate(SITE_ORDER):
        path = BASE / f"Proc_DRIAMS-{site}" / drug / "data.csv"
        df = pd.read_csv(path)
        bin_cols = [c for c in df.columns if c.startswith("bin_")]
        codes.append(df["code"].values)
        Xs.append(df[bin_cols].to_numpy(dtype="float32"))
        ys.append(df["label"].to_numpy(dtype="int64"))
        sps.append(df["species"].values)
        sites.append(np.full(len(df), si, dtype=int))
    return (np.concatenate(codes), np.concatenate(Xs, axis=0),
            np.concatenate(ys), np.concatenate(sps), np.concatenate(sites))

raw = {}
for drug in ALL_DRUGS:
    raw[drug] = load_drug(drug)
    print(f"  {drug}: {len(raw[drug][1])} samples")

print(f"Total spectra (pre-dedup): {sum(len(raw[d][1]) for d in ALL_DRUGS)}")


In [ ]:
# ── Preprocessing helper: log1p + standardize ──
def preprocess(X):
    Xl = np.log1p(X.astype("float32"))
    return StandardScaler().fit(Xl).transform(Xl)
print("preprocess() defined")


In [ ]:
# ── Global pooled dataset (dedup by isolate code) ──
# The same isolate is tested against multiple drugs -> same "code" -> dedup once.
code_all = np.concatenate([raw[d][0] for d in ALL_DRUGS])
X_all    = np.concatenate([raw[d][1] for d in ALL_DRUGS], axis=0)
sp_all   = np.concatenate([raw[d][3] for d in ALL_DRUGS])
site_all = np.concatenate([raw[d][4] for d in ALL_DRUGS])

_, first_idx = np.unique(code_all, return_index=True)
first_idx.sort()
X_g_raw   = X_all[first_idx]
species_g = sp_all[first_idx]
site_g    = site_all[first_idx]
print(f"Global: {len(code_all)} spectra -> {len(first_idx)} unique isolates")
print(f"  per site: {np.bincount(site_g)}")
print(f"  species: {len(np.unique(species_g))}")

X_g = preprocess(X_g_raw)
print("Global preprocessed:", X_g.shape)


In [ ]:
# ── Subsampling + UMAP fit helpers ──
def subsample_idx(labels, n, seed=SEED):
    n = int(min(n, len(labels)))
    if n == len(labels):
        return np.arange(len(labels))
    rng = np.random.default_rng(seed)
    chosen = []
    for u in np.unique(labels):
        gi = np.where(labels == u)[0]
        k = max(1, int(round(n * len(gi) / len(labels))))
        k = min(k, len(gi))
        chosen.append(rng.choice(gi, size=k, replace=False))
    sel = np.concatenate(chosen)
    if len(sel) > n:
        sel = rng.choice(sel, size=n, replace=False)
    return np.sort(sel)

def fit_umap(X):
    reducer = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        n_components=UMAP_N_COMPONENTS,
        random_state=UMAP_RANDOM_STATE,
        metric="euclidean")
    return reducer.fit_transform(X)

gs = subsample_idx(site_g, GLOBAL_SUBSAMPLE)
site_gs = site_g[gs]
species_gs = species_g[gs]
print(f"Global UMAP will run on {len(gs)} subsampled isolates")
emb_g = fit_umap(X_g[gs])
print("Global UMAP embedding:", emb_g.shape)


In [ ]:
# ── Global UMAP plot: site + species ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
for i, s in enumerate(SITE_ORDER):
    m = site_gs == i
    ax.scatter(emb_g[m, 0], emb_g[m, 1], s=4, alpha=0.55, label=s)
ax.set_title("Global UMAP — site", fontsize=12, fontweight="bold")
ax.legend(markerscale=5, fontsize=9); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")

ax = axes[1]
top_sp = [s for s, _ in Counter(species_gs).most_common(12)]
other = ~np.isin(species_gs, top_sp)
for sp in top_sp:
    m = species_gs == sp
    ax.scatter(emb_g[m, 0], emb_g[m, 1], s=4, alpha=0.55, label=sp)
ax.scatter(emb_g[other, 0], emb_g[other, 1], s=4, alpha=0.25, c="grey", label="other")
ax.set_title("Global UMAP — species (top-12)", fontsize=12, fontweight="bold")
ax.legend(fontsize=6, markerscale=3); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")

plt.tight_layout()
plt.savefig(OUT_DIR / "global_umap.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ── Per-drug UMAP grid: site + resistance ──
n_drugs = len(ALL_DRUGS)
fig, axes = plt.subplots(n_drugs, 2, figsize=(13, 4 * n_drugs))
for r, drug in enumerate(ALL_DRUGS):
    _, X_d, y_d, _, site_d = raw[drug]
    Xd = preprocess(X_d)
    ds = subsample_idx(site_d, PER_DRUG_SUBSAMPLE)
    emb = fit_umap(Xd[ds])

    ax = axes[r, 0]
    for i, s in enumerate(SITE_ORDER):
        m = site_d[ds] == i
        ax.scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.55, label=s)
    ax.set_title(f"{drug} — site", fontsize=10); ax.legend(fontsize=7, markerscale=4)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")

    ax = axes[r, 1]
    for lab, c, name in [(0, "#1f77b4", "S"), (1, "#d62728", "R")]:
        m = y_d[ds] == lab
        ax.scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.55, color=c, label=name)
    ax.set_title(f"{drug} — resistance", fontsize=10); ax.legend(fontsize=8)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")

plt.tight_layout()
plt.savefig(OUT_DIR / "per_drug_umap_grid.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ── Load shared masks (species masks, drug-independent) ──
def load_mask(name):
    p = SHARED_MASKS_DIR / name
    if p.exists():
        return np.load(p).astype(int)
    print(f"  [warn] {name} not found")
    return np.array([], dtype=int)

union_mask    = load_mask("union_mask.npy")
majority_mask = load_mask("majority_mask.npy")
persite_masks = {s: load_mask(f"persite_site_{s}_mask.npy") for s in SITE_ORDER}

metadata = {}
mp = SHARED_MASKS_DIR / "metadata.json"
if mp.exists():
    with open(mp) as f:
        metadata = json.load(f)

print(f"union={len(union_mask)}  majority={len(majority_mask)}")
print("persite:", {s: len(v) for s, v in persite_masks.items()})
print("metadata mask_sizes:", metadata.get("mask_sizes"))


In [ ]:
# ── Mask-overlay UMAP: does site separation persist after zeroing mask bins? ──
def apply_mask(X, mask):
    Xc = X.copy()
    if len(mask) > 0:
        Xc[:, mask] = 0.0
    return Xc

overlays = [("unmasked", None), ("majority", majority_mask), ("union", union_mask)]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (name, mask) in zip(axes, overlays):
    Xv = X_g if mask is None else apply_mask(X_g, mask)
    emb = fit_umap(Xv[gs])
    for i, s in enumerate(SITE_ORDER):
        m = site_gs == i
        ax.scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.55, label=s)
    ax.set_title(f"site — {name}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, markerscale=4); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")

plt.tight_layout()
plt.savefig(OUT_DIR / "mask_overlay_umap.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ── Quantitative companion: site separability before/after masking ──
def site_metrics(X, site_labels, n_sil=2500, n_knn=5000):
    sel_sil = subsample_idx(site_labels, n_sil)
    sil = silhouette_score(X[sel_sil], site_labels[sel_sil])
    sel_knn = subsample_idx(site_labels, n_knn)
    knn = KNeighborsClassifier(n_neighbors=5)
    acc = cross_val_score(knn, X[sel_knn], site_labels[sel_knn], cv=5,
                          scoring="accuracy", n_jobs=-1).mean()
    return sil, acc

rows = []
for name, mask in overlays:
    Xv = X_g if mask is None else apply_mask(X_g, mask)
    sil, acc = site_metrics(Xv, site_g)
    rows.append({"mask": name, "silhouette_site": sil, "knn5_site_acc": acc})
    print(f"  {name:10s}: silhouette={sil:.4f}  kNN-5 site acc={acc:.4f}")

df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(OUT_DIR / "umap_metrics.csv", index=False)
print("\numap_metrics.csv saved")


In [ ]:
# ── Summary ──
print("\n" + "=" * 60)
print(f"  UMAP diagnostic complete. {len(ALL_DRUGS)} drugs analyzed.")
print(f"  Results in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}")

print("\nInterpretation targets:")
print("  - Global UMAP: do sites (esp. D) form separate clusters?")
print("  - Mask overlay: does site separation persist after zeroing species masks?")
print("  - Per-drug: is resistance entangled with site, and is D an outlier per drug?")
